# Purpose
This notebook demonstrates how a single simulation instance is generated for a representative parameter setting. The large-scale simulations used for the analyses in the manuscript were generated using the corresponding HPC scripts, which run multiple simulation instances in parallel.

In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
from pathlib import Path
import sys
sys.path.append('..')
from scripts import weight_adjustment, rewire, basic_components as bcomp, draw

In [4]:
cond_ls = ['A', 'C'] # Weight adjustment condition
cond_color = ['hotpink', 'c'] # Color of histograms
cond_label = ['A-in', 'C-out'] # Title of histograms

eta = 10.0 # learning rate
steps = 10000 # sample interval, save adjacency matrix every 'steps' steps of weight adjustment
n = 100 # number of nodes

k = 0 # index of representative networks

In [ ]:
# Create the output directory if it does not exist
output_dir = Path("Representative networks")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Generate an initial network
edges = int(np.round(2 * np.log(n) * (n - 1), decimals=0))
initial_matrix = bcomp. generate_rand_adj(n, edges, weight_distribution= 'normal', mu=1.0, sig=0.25)

In [ ]:
f = open('./Representative networks/initial_network.pckl', 'wb')
pickle.dump(initial_matrix, f)
f.close()

# Control condition 1: adaptive weight adjustment with fixed connectivity

In [ ]:
tau = 0.1 # tau_reweight
blocks = 20 # run 'blocks * steps' steps of weight adjustment

## A-in

In [ ]:
A_matrices = {}
A_matrices[0] = initial_matrix.copy()
trunc_num = 0

A = A_matrices[0].copy()
for i in range(blocks):
    u_series = np.random.choice(n, steps)
    for u in u_series:
        A, trunc_temp = weight_adjustment.weight_adjust_Ain(A, eta, tau, u)
        trunc_num = trunc_num + trunc_temp
        A_matrices[(1+i)*steps] = A.copy()

f = open('./Representative networks/wAdjust_alone_A_tau_'+str(tau)+'.pckl', 'wb')
pickle.dump([A_matrices, trunc_num], f)
f.close()

## C-out

In [ ]:
A_matrices = {}
A_matrices[0] = initial_matrix.copy()
trunc_num = 0

A = A_matrices[0].copy()
for i in range(blocks):
    u_series = np.random.choice(n, steps)
    for u in u_series:
        A, trunc_temp = weight_adjustment.weight_adjust_Cout(A, eta, tau, u)
        trunc_num = trunc_num + trunc_temp
        A_matrices[(1+i)*steps] = A.copy()

f = open('./Representative networks/wAdjust_alone_C_tau_'+str(tau)+'.pckl', 'wb')
pickle.dump([A_matrices, trunc_num], f)
f.close()

## Histogram

In [ ]:
plt.rcParams["figure.figsize"] = (6, 2.5)
fig, ax1 = plt.subplots(1, 2)
J = blocks * steps

for s, cond in enumerate(cond_ls):
    f = open('./Representative networks/wAdjust_alone_'+cond+'_tau_'+str(tau)+'.pckl', 'rb')
    A_matrices, _ = pickle.load(f)
    f.close()
    # Histogram of the representative networks
    weights = A_matrices[J][A_matrices[J]>0]
    ax1[s].hist(weights, bins=30, density=True,color = cond_color[s], alpha=0.7)
    ax1[s].set_title(cond_label[s], fontsize = 13)
    ax1[s].set_xlabel('Weights',fontsize=10)
    ax1[s].tick_params(axis = 'both', labelsize=10)
    ax1[s].set_xlim([-0.5, 13])
    ax1[s].set_xlabel('Weights',fontsize=10)
    # histogram of log(weights)
    left, bottom, width, height = [0.265 + s * 0.435, 0.5, 0.18, 0.35]
    ax1_1 = fig.add_axes([left, bottom, width, height])
    x, mu, sig = draw.hist_log_weight(ax1_1, weights, num_bins = 30, color = cond_color[s])
ax1[0].set_ylabel('Density',fontsize=12)
fig.subplots_adjust(wspace = 0.3)

# Control condition 2: adaptive weight adjustment with random rewiring

In [ ]:
tau = 0.1 # tau_reweight
tau_rewire = 1.0 # tau_rewire, used to match the steps of adaptive rewiring to the dual adaptive algorithm
rate = int(tau_rewire / tau) # rewire evert 'rate' steps of weight adjustment
blocks = 20 # run 'blocks * steps' steps of weight adjustment

## A-in

In [ ]:
A_matrices = {}
A_matrices[0] = initial_matrix.copy()
trunc_num = 0

A = A_matrices[0].copy()
for i in range(blocks*steps):
    u = np.random.choice(n)
    A, trunc_temp = weight_adjustment.weight_adjust_Ain(A, eta, tau, u)
    trunc_num = trunc_num + trunc_temp
        
    if (i+1)%rate==0:
        r = np.random.random_sample() # choose incoming or outgoing connections
        if r<0.5:
            flag = 'in'
        else:
            flag = 'out'
        A = rewire.random_rewire(A,  n, flag)
            
    if (i+1)%steps == 0:
        A_matrices[1+i] = A.copy()

f = open('./Representative networks/wAdjust_randRew_A_tau_'+str(tau)+'.pckl', 'wb')
pickle.dump([A_matrices, trunc_num], f)
f.close()

## C-out

In [ ]:
A_matrices = {}
A_matrices[0] = initial_matrix.copy()
trunc_num = 0

A = A_matrices[0].copy()
for i in range(blocks*steps):
    u = np.random.choice(n)
    A, trunc_temp = weight_adjustment.weight_adjust_Cout(A, eta, tau, u)
    trunc_num = trunc_num + trunc_temp
        
    if (i+1)%rate==0:
        r = np.random.random_sample() # choose incoming or outgoing connections
        if r<0.5:
            flag = 'in'
        else:
            flag = 'out'
        A = rewire.random_rewire(A,  n, flag)
            
    if (i+1)%steps == 0:
        A_matrices[1+i] = A.copy()

f = open('./Representative networks/wAdjust_randRew_C_tau_'+str(tau)+'.pckl', 'wb')
pickle.dump([A_matrices, trunc_num], f)
f.close()

## Histogram

In [ ]:
plt.rcParams["figure.figsize"] = (6, 2.5)
fig, ax1 = plt.subplots(1, 2)
cond_color = ['hotpink', 'c']; cond_label = ['A-in', 'C-out']
J = blocks * steps
for s, cond in enumerate(cond_ls):
    f = open('./Representative networks/wAdjust_randRew_'+cond+'_tau_'+str(tau)+'.pckl', 'rb')
    A_matrices, _ = pickle.load(f)
    f.close()
    # Histogram of the representative networks
    weights = A_matrices[J][A_matrices[J]>0]
    ax1[s].hist(weights, bins=30, density=True,color = cond_color[s], alpha=0.7)
    ax1[s].set_title(cond_label[s], fontsize = 13)
    ax1[s].set_xlabel('Weights',fontsize=10)
    ax1[s].tick_params(axis = 'both', labelsize=10)
    ax1[s].set_xlim([-0.5, 13])
    ax1[s].set_xlabel('Weights',fontsize=10)
    # histogram of log(weights)
    left, bottom, width, height = [0.265 + s * 0.435, 0.5, 0.18, 0.35]
    ax1_1 = fig.add_axes([left, bottom, width, height])
    x, mu, sig = draw.hist_log_weight(ax1_1, weights, num_bins = 30, color = cond_color[s])
ax1[0].set_ylabel('Density',fontsize=12)
fig.subplots_adjust(wspace = 0.3)

# Dual adaptive: adaptive weight adjustment with adaptive rewiring

In [ ]:
tau = 0.1 # tau_reweight
p_in = 0.5 # proportion of in-link rewiring
tau_rewire = 1.0 # tau_rewire
rate = int(tau_rewire / tau) # rewire evert 'rate' steps of weight adjustment
blocks = 100 # run 'blocks * steps' steps of weight adjustment

## A-in

In [ ]:
A_matrices = {}
A_matrices[0] = initial_matrix.copy()
trunc_num = 0

A = A_matrices[0].copy()
for i in range(blocks*steps):
    u = np.random.choice(n)
    A, trunc_temp = weight_adjustment.weight_adjust_Ain(A, eta, tau, u)
    trunc_num = trunc_num + trunc_temp
        
    if (i+1)%rate==0:
        r = np.random.random_sample() # choose incoming or outgoing connections
        if r<p_in:
            flag = 'in'
        else:
            flag = 'out'
            
        conK = bcomp.consensus_kernel(A, tau_rewire)
        advK = bcomp.advection_kernel(A, tau_rewire)
        A = rewire.rewire(A, conK, advK, n, flag)
            
    if (i+1)%steps == 0:
        A_matrices[1+i] = A.copy()
f = open('./Representative networks/dualAdapt_A_tau_'+str(tau)+'_pin_'+str(p_in)+'.pckl', 'wb')
pickle.dump([A_matrices, trunc_num], f)
f.close()

## C-out

In [ ]:
A_matrices = {}
A_matrices[0] = initial_matrix.copy()
trunc_num = 0

A = A_matrices[0].copy()
for i in range(blocks*steps):
    u = np.random.choice(n)
    A, trunc_temp = weight_adjustment.weight_adjust_Cout(A, eta, tau, u)
    trunc_num = trunc_num + trunc_temp
        
    if (i+1)%rate==0:
        r = np.random.random_sample() # choose incoming or outgoing connections
        if r<p_in:
            flag = 'in'
        else:
            flag = 'out'
            
        conK = bcomp.consensus_kernel(A, tau_rewire)
        advK = bcomp.advection_kernel(A, tau_rewire)
        A = rewire.rewire(A, conK, advK, n, flag)
            
    if (i+1)%steps == 0:
        A_matrices[1+i] = A.copy()
f = open('./Representative networks/dualAdapt_C_tau_'+str(tau)+'_pin_'+str(p_in)+'.pckl', 'wb')
pickle.dump([A_matrices, trunc_num], f)
f.close()

## Histogram

In [ ]:
plt.rcParams["figure.figsize"] = (6, 2.5)
fig, ax1 = plt.subplots(1, 2)
cond_color = ['hotpink', 'c']; cond_label = ['A-in', 'C-out']
J = blocks * steps
for s, cond in enumerate(cond_ls):
    f = open('./Representative networks/dualAdapt_'+cond+'_tau_'+str(tau)+'_pin_'+str(p_in)+'.pckl', 'rb')
    A_matrices, _ = pickle.load(f)
    f.close()
    # Histogram of the representative networks
    weights = A_matrices[J][A_matrices[J]>0]
    ax1[s].hist(weights, bins=30, density=True,color = cond_color[s], alpha=0.7)
    ax1[s].set_title(cond_label[s], fontsize = 13)
    ax1[s].set_xlabel('Weights',fontsize=10)
    ax1[s].tick_params(axis = 'both', labelsize=10)
    ax1[s].set_xlim([-0.5, 13])
    ax1[s].set_xlabel('Weights',fontsize=10)
    # histogram of log(weights)
    left, bottom, width, height = [0.265 + s * 0.435, 0.5, 0.18, 0.35]
    ax1_1 = fig.add_axes([left, bottom, width, height])
    x, mu, sig = draw.hist_log_weight(ax1_1, weights, num_bins = 30, color = cond_color[s])
ax1[0].set_ylabel('Density',fontsize=12)
fig.subplots_adjust(wspace = 0.3)

# CD units setting: dual adaptive + a small proportion of random rewiring

In [ ]:
tau = 0.1 # tau_reweight
p_in = 0.5 # proportion of in-link rewiring
prand = 0.1 # proportion of random rewiring
tau_rewire = 1.0 # tau_rewire
rate = int(tau_rewire / tau) # rewire evert 'rate' steps of weight adjustment
blocks = 50 # run 'blocks * steps' steps of weight adjustment

## A-in

In [ ]:
A_matrices = {}
A_matrices[0] = initial_matrix.copy()
trunc_num = 0

A = A_matrices[0].copy()
for i in range(blocks*steps):
    u = np.random.choice(n)
    A, trunc_temp = weight_adjustment.weight_adjust_Ain(A, eta, tau, u)
    trunc_num = trunc_num + trunc_temp
        
    if (i+1)%rate==0:
        r1 = np.random.random_sample() # choose incoming or outgoing connections
        if r1<p_in:
            flag = 'in'
        else:
            flag = 'out'
        r2 = np.random.random_sample() # choose random or adaptive rewiring
        if r2<prand:
            A = rewire.random_rewire(A, n, flag)
        else:
            conK = bcomp.consensus_kernel(A, tau_rewire)
            advK = bcomp.advection_kernel(A, tau_rewire)
            A = rewire.rewire(A, conK, advK, n, flag)
            
    if (i+1)%steps == 0:
        A_matrices[1+i] = A.copy()
f = open('./Representative networks/cdUnits_A_tau_'+str(tau)+'_prand_'+str(prand)+'.pckl', 'wb')
pickle.dump([A_matrices, trunc_num], f)
f.close()

## C-out

In [ ]:
A_matrices = {}
A_matrices[0] = initial_matrix.copy()
trunc_num = 0

A = A_matrices[0].copy()
for i in range(blocks*steps):
    u = np.random.choice(n)
    A, trunc_temp = weight_adjustment.weight_adjust_Cout(A, eta, tau, u)
    trunc_num = trunc_num + trunc_temp
        
    if (i+1)%rate==0:
        r1 = np.random.random_sample() # choose incoming or outgoing connections
        if r1<p_in:
            flag = 'in'
        else:
            flag = 'out'
        r2 = np.random.random_sample() # choose random or adaptive rewiring
        if r2<prand:
            A = rewire.random_rewire(A, n, flag)
        else:
            conK = bcomp.consensus_kernel(A, tau_rewire)
            advK = bcomp.advection_kernel(A, tau_rewire)
            A = rewire.rewire(A, conK, advK, n, flag)
            
    if (i+1)%steps == 0:
        A_matrices[1+i] = A.copy()
f = open('./Representative networks/cdUnits_C_tau_'+str(tau)+'_prand_'+str(prand)+'.pckl', 'wb')
pickle.dump([A_matrices, trunc_num], f)
f.close()

## Histogram

In [ ]:
plt.rcParams["figure.figsize"] = (6, 2.5)
fig, ax1 = plt.subplots(1, 2)
cond_color = ['hotpink', 'c']; cond_label = ['A-in', 'C-out']
J = blocks * steps
for s, cond in enumerate(cond_ls):
    f = open('./Representative networks/cdUnits_'+cond+'_tau_'+str(tau)+'_prand_'+str(prand)+'.pckl', 'rb')
    A_matrices, _ = pickle.load(f)
    f.close()
    # Histogram of the representative networks
    weights = A_matrices[J][A_matrices[J]>0]
    ax1[s].hist(weights, bins=30, density=True,color = cond_color[s], alpha=0.7)
    ax1[s].set_title(cond_label[s], fontsize = 13)
    ax1[s].set_xlabel('Weights',fontsize=10)
    ax1[s].tick_params(axis = 'both', labelsize=10)
    ax1[s].set_xlim([-0.5, 13])
    ax1[s].set_xlabel('Weights',fontsize=10)
    # histogram of log(weights)
    left, bottom, width, height = [0.265 + s * 0.435, 0.5, 0.18, 0.35]
    ax1_1 = fig.add_axes([left, bottom, width, height])
    x, mu, sig = draw.hist_log_weight(ax1_1, weights, num_bins = 30, color = cond_color[s])
ax1[0].set_ylabel('Density',fontsize=12)
fig.subplots_adjust(wspace = 0.3)